In [9]:
import geopandas as gpd
import numpy as np
import os, re
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor, AdaBoostRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split, RepeatedKFold, cross_validate
from sklearn.metrics import make_scorer, r2_score, mean_squared_error

# === 1）先挑一个代表性格网文件 & target ===
grid_folder = r'D:\seoul\grids\lst_map'
# 这里取第一个 _clean.shp 文件，您也可以手动指定一个您最常关注的 grid_size
filename = next(f for f in os.listdir(grid_folder) if f.endswith('city2020_lst_ratio_grid_450m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy_clean.shp'))
gdf = gpd.read_file(os.path.join(grid_folder, filename))

target_vars = ['nor_2020', 'ext_2020', 'hr_2020']  # 您也可以循环测试每个 target
explanatory = ['BCR(%)','BHV','NDVI','SVF','EV(m)','Dist_BP','Dist_MT','Dist_WB','WR(%)']

# 丢弃 NA
gdf = gdf.replace([np.inf, -np.inf], np.nan)\
         .dropna(subset=[target]+explanatory)

X = gdf[explanatory]
y = gdf[target]

# 为了更快：抽取 30% 子样本（可选）
X, _, y, _ = train_test_split(X, y, train_size=0.3, random_state=0)

# === 2）定义要比较的模型字典 ===
models = {
    # 您已明确的 GBDT 参数
    'GBDT': GradientBoostingRegressor(
        n_estimators=4168,
        learning_rate=0.018,
        max_depth=13,
        subsample=0.839,
        random_state=0
    ),

    # —— 为 RandomForest 指定常用调优参数 —— #
    'RF':  RandomForestRegressor(
    n_estimators=983,       # Ranger 的 num.trees Tun.O
    bootstrap=False,        # Ranger 的 replace Tun.O     # Ranger 的 sample.fraction Tun.O
    max_features=0.692,     # Ranger 的 mtry Tun.O (比例) 或 max_features=9
    min_samples_split=2,
    min_samples_leaf=1,     # Ranger 的 min.node.size Tun.O
    random_state=0
    ),

    # —— 为 ANN（MLPRegressor）指定调优参数 —— #
    'ANN': MLPRegressor(
        hidden_layer_sizes=(150, 100),  # 双层结构：150 → 100
        activation='relu',
        solver='adam',
        learning_rate_init=0.01,        # 略高于默认 0.001
        alpha=1e-3,                     # L2 正则（可调）
        batch_size=64,
        max_iter=1000,
        early_stopping=True,
        n_iter_no_change=20,
        tol=1e-4,
        random_state=0
    ),

    # —— 为 AdaBoost 指定调优参数 —— #
    'AdaBoost': AdaBoostRegressor(
        n_estimators=200,      # 比默认 50 更多，提高拟合能力
        learning_rate=0.5,     # 比默认 1.0 略低，防止过拟合
        random_state=0
    )
}

# === 4）循环比较 ===
results = {}
for name, mdl in models.items():
    cv_res = cross_validate(
        mdl, X, y,
        cv=cv,
        scoring=scoring,
        return_train_score=True,
        n_jobs=-1
    )
    # 验证集分数
    test_r2   = cv_res['test_R2']
    test_rmse = -cv_res['test_RMSE']
    # 训练集分数
    train_r2   = cv_res['train_R2']
    train_rmse = -cv_res['train_RMSE']

    results[name] = {
        'Train R2 (mean)':  np.mean(train_r2),
        'Train R2 (std)':   np.std(train_r2),
        'Test R2 (mean)':   np.mean(test_r2),
        'Test R2 (std)':    np.std(test_r2),
        'Train RMSE (mean)': np.mean(train_rmse),
        'Train RMSE (std)':  np.std(train_rmse),
        'Test RMSE (mean)':  np.mean(test_rmse),
        'Test RMSE (std)':   np.std(test_rmse),
    }

# === 5）展示比较结果 ===
import pandas as pd
df_cmp = pd.DataFrame(results).T
print(df_cmp)


          Train R2 (mean)  Train R2 (std)  Test R2 (mean)  Test R2 (std)  \
GBDT             1.000000        0.000000        0.870910       0.023364   
RF               1.000000        0.000000        0.872684       0.024211   
ANN             -1.281894        1.612670       -1.782553       1.564222   
AdaBoost         0.887583        0.002789        0.848236       0.014965   

          Train RMSE (mean)  Train RMSE (std)  Test RMSE (mean)  \
GBDT           1.455238e-08      1.537952e-11          1.180784   
RF             0.000000e+00      0.000000e+00          1.172115   
ANN            4.704463e+00      1.736756e+00          5.239901   
AdaBoost       1.111850e+00      1.073735e-02          1.284801   

          Test RMSE (std)  
GBDT             0.096581  
RF               0.096419  
ANN              1.562528  
AdaBoost         0.071934  


In [10]:
df_cmp

,Train R2 (mean),Train R2 (std),Test R2 (mean),Test R2 (std),Train RMSE (mean),Train RMSE (std),Test RMSE (mean),Test RMSE (std)
GBDT,1.000000,0.000000,0.870910,0.023364,1.455238e-08,1.537952e-11,1.180784,0.096581
RF,1.000000,0.000000,0.872684,0.024211,0.000000e+00,0.000000e+00,1.172115,0.096419
ANN,-1.281894,1.612670,-1.782553,1.564222,4.704463e+00,1.736756e+00,5.239901,1.562528
AdaBoost,0.887583,0.002789,0.848236,0.014965,1.111850e+00,1.073735e-02,1.284801,0.071934
